In [3]:
!nvidia-smi


Mon Aug 17 05:00:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
!pip install -q huggingface_hub

In [5]:
!mkdir -p kernels


In [6]:
%%writefile data.py

import json
import struct
import os
import torch

DEFAULT_REPO = "NousResearch/Meta-Llama-3-8B" 
CACHE_DIR = os.environ.get("INT8_GEMM_CACHE", "/kaggle/working/weight_cache")

def _shard_and_offsets(repo, filename, key):
    from huggingface_hub import hf_hub_url
    import requests

    url = hf_hub_url(repo, filename)
    r = requests.get(url, headers={"Range": "bytes=0-7"}, timeout=30)
    r.raise_for_status()
    header_len = struct.unpack("<Q", r.content)[0]

    r2 = requests.get(url, headers={"Range": f"bytes=8-{8 + header_len - 1}"}, timeout=30)
    r2.raise_for_status()
    header = json.loads(r2.content)
    meta = header[key]
    base = 8 + header_len
    start, end = meta["data_offsets"]
    return url, base + start, base + end, meta["shape"], meta["dtype"]

def _fetch_tensor(repo, filename, key, cache_path):
    import requests

    if cache_path and os.path.exists(cache_path):
        return torch.load(cache_path)

    url, start, end, shape, dtype = _shard_and_offsets(repo, filename, key)
    assert dtype == "BF16", f"expected BF16, got {dtype}"

    r = requests.get(url, headers={"Range": f"bytes={start}-{end - 1}"}, timeout=120)
    r.raise_for_status()
    t = torch.frombuffer(bytearray(r.content), dtype=torch.bfloat16).reshape(shape).clone()

    if cache_path:
        os.makedirs(os.path.dirname(cache_path), exist_ok=True)
        torch.save(t, cache_path)
    return t

def load_llama_down_proj(layer=0, repo=DEFAULT_REPO, use_cache=True):

    key = f"model.layers.{layer}.mlp.down_proj.weight"
    cache_path = os.path.join(CACHE_DIR, f"down_proj_layer{layer}.pt") if use_cache else None

    try:
        from huggingface_hub import hf_hub_download
        index_path = hf_hub_download(repo, "model.safetensors.index.json")
        weight_map = json.load(open(index_path))["weight_map"]
        filename = weight_map[key]

        W_bf16 = _fetch_tensor(repo, filename, key, cache_path)
        W = W_bf16.to(torch.float16)
        return W, f"real:{repo}:{key}"
    except Exception as e:
        print(f"[data.py] Could not fetch real weights ({type(e).__name__}: {e}). "
              f"Falling back to synthetic N(0, 0.02) weight of the same shape. "
              f"Metrics computed on synthetic weights are NOT representative -- "
              f"re-run with internet access before trusting the numbers.")
        torch.manual_seed(0)
        W = (torch.randn(4096, 14336) * 0.02).half()
        return W, "synthetic:randn(0,0.02)"


if __name__ == "__main__":
    W, source = load_llama_down_proj()
    print(f"source: {source}")
    print(f"shape: {tuple(W.shape)}, dtype: {W.dtype}")
    print(f"abs max: {W.abs().max().item():.4f}, abs mean: {W.abs().mean().item():.6f}")

Writing data.py


In [7]:

%%writefile metrics.py
import torch
def cosine_similarity(ref: torch.Tensor, approx: torch.Tensor) -> float:
    a = ref.flatten().float()
    b = approx.flatten().float()
    return (torch.dot(a, b) / (a.norm() * b.norm())).item()
def relative_l2_error(ref: torch.Tensor, approx: torch.Tensor) -> float:
    ref = ref.float()
    approx = approx.float()
    return ((approx - ref).norm() / ref.norm()).item()

def max_abs_error(ref: torch.Tensor, approx: torch.Tensor) -> float:
    return (approx.float() - ref.float()).abs().max().item()

def mean_abs_error(ref: torch.Tensor, approx: torch.Tensor) -> float:
    return (approx.float() - ref.float()).abs().mean().item()

def full_report(ref: torch.Tensor, approx: torch.Tensor) -> dict:
    ref_mag = ref.float().abs().mean().item()
    mae = mean_abs_error(ref, approx)
    return {
        "cosine_similarity": cosine_similarity(ref, approx),
        "relative_l2_error": relative_l2_error(ref, approx),
        "max_abs_error": max_abs_error(ref, approx),
        "mean_abs_error": mae,
        "mean_abs_error_pct_of_output_magnitude": mae / ref_mag if ref_mag > 0 else float("nan"),
    }

def print_report(name: str, ref: torch.Tensor, approx: torch.Tensor):
    r = full_report(ref, approx)
    print(f"  [{name}] cos_sim={r['cosine_similarity']:.4f}  "
          f"rel_L2={r['relative_l2_error']:.4%}  "
          f"max_abs_err={r['max_abs_error']:.4f}  "
          f"mean_abs_err={r['mean_abs_error']:.5f} "
          f"({r['mean_abs_error_pct_of_output_magnitude']:.2%} of output magnitude)")

Writing metrics.py


In [8]:
%%writefile quantize.py

import torch
def quantize_weight_per_channel(W: torch.Tensor):
    """W: [out_features, in_features] fp16. Returns (W_int8, scale[out_features])."""
    amax = W.float().abs().amax(dim=1, keepdim=True).clamp(min=1e-8)
    scale = amax / 127.0
    W_int8 = torch.round(W.float() / scale).clamp(-127, 127).to(torch.int8)
    return W_int8, scale.squeeze(1).half()


def quantize_activation_dynamic(X: torch.Tensor):
    """X: [batch, in_features] fp16. Returns (X_int8, scale scalar)."""
    amax = X.float().abs().amax().clamp(min=1e-8)
    scale = amax / 127.0
    X_int8 = torch.round(X.float() / scale).clamp(-127, 127).to(torch.int8)
    return X_int8, scale.half()


def dequantize(acc_int32: torch.Tensor, x_scale: torch.Tensor, w_scale: torch.Tensor):
    """acc_int32: [M, out_features] int32. w_scale: [out_features]."""
    return acc_int32.float() * x_scale.float() * w_scale.float().unsqueeze(0)


if __name__ == "__main__":
    torch.manual_seed(0)
    W = (torch.randn(16, 32) * 0.02).half()
    Wq, w_scale = quantize_weight_per_channel(W)
    assert Wq.dtype == torch.int8 and w_scale.shape == (16,)

    X = (torch.randn(4, 32) * 0.5).half()
    Xq, x_scale = quantize_activation_dynamic(X)
    assert Xq.dtype == torch.int8 and x_scale.dim() == 0

    acc = Xq.to(torch.int32) @ Wq.to(torch.int32).t()
    Y = dequantize(acc, x_scale, w_scale)
    Y_ref = (X.float() @ W.float().t())
    err = (Y - Y_ref).abs().mean().item()
    print(f"self-test mean abs error on random [16,32] weight: {err:.6f}  (sanity check only)")

Writing quantize.py


In [9]:
%%writefile kernels/int8_gemm_v1.cu

#include <torch/extension.h>
#include <cuda_runtime.h>

#define TILE 16

__global__ void int8_gemm_v1_kernel(
    const int8_t* __restrict__ X,   // [M, K]
    const int8_t* __restrict__ W,   // [N, K]
    const float* __restrict__ w_scale,   // [N]
    float x_scale,
    float* __restrict__ Y,          // [M, N]
    int M, int N, int K)
{
    __shared__ int8_t X_tile[TILE][TILE];
    __shared__ int8_t W_tile[TILE][TILE];

    int row = blockIdx.y * TILE + threadIdx.y;   // index into M
    int col = blockIdx.x * TILE + threadIdx.x;   // index into N

    int32_t acc = 0;

    for (int t = 0; t < (K + TILE - 1) / TILE; ++t) {
        int k_x = t * TILE + threadIdx.x;
        int k_w = t * TILE + threadIdx.x;

        X_tile[threadIdx.y][threadIdx.x] =
            (row < M && k_x < K) ? X[row * K + k_x] : 0;
        // W is [N, K]; each thread loads one element of the row `col`
        W_tile[threadIdx.y][threadIdx.x] =
            (col < N && k_w < K) ? W[col * K + (t * TILE + threadIdx.y)] : 0;

        __syncthreads();

        #pragma unroll
        for (int k = 0; k < TILE; ++k) {
            acc += static_cast<int32_t>(X_tile[threadIdx.y][k]) *
                   static_cast<int32_t>(W_tile[k][threadIdx.x]);
        }

        __syncthreads();
    }

    if (row < M && col < N) {
        Y[row * N + col] = static_cast<float>(acc) * x_scale * w_scale[col];
    }
}

torch::Tensor int8_gemm_v1(
    torch::Tensor X,        // [M, K] int8
    torch::Tensor W,        // [N, K] int8
    torch::Tensor w_scale,  // [N] float32
    torch::Tensor x_scale)  // scalar float32
{
    TORCH_CHECK(X.is_cuda() && W.is_cuda(), "inputs must be CUDA tensors");
    TORCH_CHECK(X.dtype() == torch::kInt8 && W.dtype() == torch::kInt8, "X, W must be int8");
    TORCH_CHECK(X.size(1) == W.size(1), "K dimension mismatch");

    int M = X.size(0);
    int K = X.size(1);
    int N = W.size(0);

    auto Y = torch::empty({M, N}, X.options().dtype(torch::kFloat32));

    dim3 block(TILE, TILE);
    dim3 grid((N + TILE - 1) / TILE, (M + TILE - 1) / TILE);

    int8_gemm_v1_kernel<<<grid, block>>>(
        X.data_ptr<int8_t>(),
        W.data_ptr<int8_t>(),
        w_scale.data_ptr<float>(),
        x_scale.item<float>(),
        Y.data_ptr<float>(),
        M, N, K);

    return Y;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("int8_gemm_v1", &int8_gemm_v1, "Naive tiled INT8 GEMM (v1)");
}

Writing kernels/int8_gemm_v1.cu


In [10]:
%%writefile kernels/int8_gemm_v2.cu
#include <torch/extension.h>
#include <cuda_runtime.h>

#define TILE 16

__global__ void int8_gemm_v2_kernel(
    const int32_t* __restrict__ X4,   // [M, K/4], each int32 = 4 packed int8
    const int32_t* __restrict__ W4,   // [N, K/4]
    const float* __restrict__ w_scale,   // [N]
    float x_scale,
    float* __restrict__ Y,          // [M, N]
    int M, int N, int K4)
{
    __shared__ int32_t X_tile[TILE][TILE];
    __shared__ int32_t W_tile[TILE][TILE];

    int row = blockIdx.y * TILE + threadIdx.y;   // index into M
    int col = blockIdx.x * TILE + threadIdx.x;   // index into N

    int32_t acc = 0;

    for (int t = 0; t < (K4 + TILE - 1) / TILE; ++t) {
        int k4 = t * TILE + threadIdx.x;

        // Coalesced: consecutive threadIdx.x -> consecutive int32 words.
        X_tile[threadIdx.y][threadIdx.x] =
            (row < M && k4 < K4) ? X4[row * K4 + k4] : 0;
        W_tile[threadIdx.y][threadIdx.x] =
            (col < N && k4 < K4) ? W4[col * K4 + (t * TILE + threadIdx.y)] : 0;

        __syncthreads();

        #pragma unroll
        for (int k = 0; k < TILE; ++k) {
            // Each call does 4 multiply-adds (one packed int32 = 4 int8s) in one instruction.
            acc = __dp4a(X_tile[threadIdx.y][k], W_tile[k][threadIdx.x], acc);
        }

        __syncthreads();
    }

    if (row < M && col < N) {
        Y[row * N + col] = static_cast<float>(acc) * x_scale * w_scale[col];
    }
}

torch::Tensor int8_gemm_v2(
    torch::Tensor X,        // [M, K] int8
    torch::Tensor W,        // [N, K] int8
    torch::Tensor w_scale,  // [N] float32
    torch::Tensor x_scale)  // scalar float32
{
    TORCH_CHECK(X.is_cuda() && W.is_cuda(), "inputs must be CUDA tensors");
    TORCH_CHECK(X.dtype() == torch::kInt8 && W.dtype() == torch::kInt8, "X, W must be int8");
    TORCH_CHECK(X.is_contiguous() && W.is_contiguous(), "X, W must be contiguous");
    TORCH_CHECK(X.size(1) == W.size(1), "K dimension mismatch");
    TORCH_CHECK(X.size(1) % 4 == 0, "K must be divisible by 4 for dp4a packing");

    int M = X.size(0);
    int K = X.size(1);
    int N = W.size(0);
    int K4 = K / 4;

    auto Y = torch::empty({M, N}, X.options().dtype(torch::kFloat32));

    dim3 block(TILE, TILE);
    dim3 grid((N + TILE - 1) / TILE, (M + TILE - 1) / TILE);

    int8_gemm_v2_kernel<<<grid, block>>>(
        reinterpret_cast<const int32_t*>(X.data_ptr<int8_t>()),
        reinterpret_cast<const int32_t*>(W.data_ptr<int8_t>()),
        w_scale.data_ptr<float>(),
        x_scale.item<float>(),
        Y.data_ptr<float>(),
        M, N, K4);

    return Y;
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("int8_gemm_v2", &int8_gemm_v2, "dp4a-vectorized tiled INT8 GEMM (v2)");
}

Writing kernels/int8_gemm_v2.cu


In [11]:
%%writefile kernels/int8_gemm_v3.cu
#include <torch/extension.h>
#include <cuda_runtime.h>
#include <mma.h>

using namespace nvcuda;

#define WMMA_M 16
#define WMMA_N 16
#define WMMA_K 16

__global__ void int8_gemm_v3_kernel(
    const int8_t* __restrict__ X,        // [M_padded, K] row-major
    const int8_t* __restrict__ W,        // [N, K] row-major == [K, N] col-major (ldm=K)
    const float* __restrict__ w_scale,   // [N]
    float x_scale,
    float* __restrict__ Y,               // [M_padded, N]
    int M_padded, int N, int K)
{
    int tile_m = blockIdx.y;   // which 16-row tile of M this block/warp owns
    int tile_n = blockIdx.x;   // which 16-col tile of N this block/warp owns
    int row0 = tile_m * WMMA_M;
    int col0 = tile_n * WMMA_N;

    wmma::fragment<wmma::matrix_a, WMMA_M, WMMA_N, WMMA_K, int8_t, wmma::row_major> a_frag;
    wmma::fragment<wmma::matrix_b, WMMA_M, WMMA_N, WMMA_K, int8_t, wmma::col_major> b_frag;
    wmma::fragment<wmma::accumulator, WMMA_M, WMMA_N, WMMA_K, int32_t> acc_frag;
    wmma::fill_fragment(acc_frag, 0);

    for (int k = 0; k < K; k += WMMA_K) {
        const int8_t* a_ptr = X + row0 * K + k;   // A tile top-left, row-major, ldm=K
        const int8_t* b_ptr = W + col0 * K + k;   // B tile top-left, col-major, ldm=K (see header note)

        wmma::load_matrix_sync(a_frag, a_ptr, K);
        wmma::load_matrix_sync(b_frag, b_ptr, K);
        wmma::mma_sync(acc_frag, a_frag, b_frag, acc_frag);
    }

    __shared__ int32_t tile_smem[WMMA_M * WMMA_N];
    wmma::store_matrix_sync(tile_smem, acc_frag, WMMA_N, wmma::mem_row_major);
    __syncthreads();

    // One warp (32 threads) writes out a 16x16=256-element tile: 8 elements/thread.
    for (int idx = threadIdx.x; idx < WMMA_M * WMMA_N; idx += 32) {
        int local_row = idx / WMMA_N;
        int local_col = idx % WMMA_N;
        int g_row = row0 + local_row;
        int g_col = col0 + local_col;
        if (g_row < M_padded && g_col < N) {
            Y[g_row * N + g_col] = static_cast<float>(tile_smem[idx]) * x_scale * w_scale[g_col];
        }
    }
}

torch::Tensor int8_gemm_v3(
    torch::Tensor X,        // [M, K] int8, M need not be a multiple of 16
    torch::Tensor W,        // [N, K] int8
    torch::Tensor w_scale,  // [N] float32
    torch::Tensor x_scale)  // scalar float32
{
    TORCH_CHECK(X.is_cuda() && W.is_cuda(), "inputs must be CUDA tensors");
    TORCH_CHECK(X.dtype() == torch::kInt8 && W.dtype() == torch::kInt8, "X, W must be int8");
    TORCH_CHECK(X.is_contiguous() && W.is_contiguous(), "X, W must be contiguous");
    TORCH_CHECK(X.size(1) == W.size(1), "K dimension mismatch");

    int M = X.size(0);
    int K = X.size(1);
    int N = W.size(0);

    TORCH_CHECK(K % WMMA_K == 0, "K must be divisible by 16 for wmma tensor-core tiles");
    TORCH_CHECK(N % WMMA_N == 0, "N must be divisible by 16 for wmma tensor-core tiles");

    int M_padded = ((M + WMMA_M - 1) / WMMA_M) * WMMA_M;

    torch::Tensor X_padded;
    if (M_padded != M) {
        X_padded = torch::zeros({M_padded, K}, X.options());
        X_padded.slice(0, 0, M).copy_(X);
    } else {
        X_padded = X;
    }

    auto Y_padded = torch::empty({M_padded, N}, X.options().dtype(torch::kFloat32));

    dim3 block(32, 1, 1);   // one warp per block; each warp owns one 16x16 output tile
    dim3 grid(N / WMMA_N, M_padded / WMMA_M);

    int8_gemm_v3_kernel<<<grid, block>>>(
        X_padded.data_ptr<int8_t>(),
        W.data_ptr<int8_t>(),
        w_scale.data_ptr<float>(),
        x_scale.item<float>(),
        Y_padded.data_ptr<float>(),
        M_padded, N, K);

    return (M_padded == M) ? Y_padded : Y_padded.slice(0, 0, M).contiguous();
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("int8_gemm_v3", &int8_gemm_v3, "Tensor-core (wmma) INT8 GEMM (v3)");
}

Writing kernels/int8_gemm_v3.cu


In [12]:
%%writefile kernels/int8_gemm_v4.cu
#include <torch/extension.h>
#include <cuda_runtime.h>
#include <mma.h>

using namespace nvcuda;

#define WMMA_M 16
#define WMMA_N 16
#define WMMA_K 16
#define NUM_WARPS 8
#define TILE_ELEMS (WMMA_M * WMMA_N)   // 256, and NUM_WARPS * 32 threads = 256 -- one thread per output element

__global__ void int8_gemm_v4_kernel(
    const int8_t* __restrict__ X,        // [M_padded, K] row-major
    const int8_t* __restrict__ W,        // [N, K] row-major == [K, N] col-major (ldm=K)
    const float* __restrict__ w_scale,   // [N]
    float x_scale,
    float* __restrict__ Y,               // [M_padded, N]
    int M_padded, int N, int K)
{
    int tile_m = blockIdx.y;
    int tile_n = blockIdx.x;
    int row0 = tile_m * WMMA_M;
    int col0 = tile_n * WMMA_N;
    int warp_id = threadIdx.y;   // 0..NUM_WARPS-1

    wmma::fragment<wmma::matrix_a, WMMA_M, WMMA_N, WMMA_K, int8_t, wmma::row_major> a_frag;
    wmma::fragment<wmma::matrix_b, WMMA_M, WMMA_N, WMMA_K, int8_t, wmma::col_major> b_frag;
    wmma::fragment<wmma::accumulator, WMMA_M, WMMA_N, WMMA_K, int32_t> acc_frag;
    wmma::fill_fragment(acc_frag, 0);

    // Strided K-split: warp `warp_id` handles k-tiles warp_id, warp_id+NUM_WARPS, ...
    // Correct regardless of whether K/WMMA_K divides evenly by NUM_WARPS.
    for (int k = warp_id * WMMA_K; k < K; k += NUM_WARPS * WMMA_K) {
        const int8_t* a_ptr = X + row0 * K + k;
        const int8_t* b_ptr = W + col0 * K + k;

        wmma::load_matrix_sync(a_frag, a_ptr, K);
        wmma::load_matrix_sync(b_frag, b_ptr, K);
        wmma::mma_sync(acc_frag, a_frag, b_frag, acc_frag);
    }

    __shared__ int32_t partial[NUM_WARPS][TILE_ELEMS];
    wmma::store_matrix_sync(partial[warp_id], acc_frag, WMMA_N, wmma::mem_row_major);
    __syncthreads();

    // 256 threads (32 * NUM_WARPS=8), 256 output elements: one thread per element.
    int tid = threadIdx.y * blockDim.x + threadIdx.x;
    int local_row = tid / WMMA_N;
    int local_col = tid % WMMA_N;
    int g_row = row0 + local_row;
    int g_col = col0 + local_col;

    int32_t sum = 0;
    #pragma unroll
    for (int w = 0; w < NUM_WARPS; ++w) {
        sum += partial[w][tid];
    }

    if (g_row < M_padded && g_col < N) {
        Y[g_row * N + g_col] = static_cast<float>(sum) * x_scale * w_scale[g_col];
    }
}

torch::Tensor int8_gemm_v4(
    torch::Tensor X,        // [M, K] int8, M need not be a multiple of 16
    torch::Tensor W,        // [N, K] int8
    torch::Tensor w_scale,  // [N] float32
    torch::Tensor x_scale)  // scalar float32
{
    TORCH_CHECK(X.is_cuda() && W.is_cuda(), "inputs must be CUDA tensors");
    TORCH_CHECK(X.dtype() == torch::kInt8 && W.dtype() == torch::kInt8, "X, W must be int8");
    TORCH_CHECK(X.is_contiguous() && W.is_contiguous(), "X, W must be contiguous");
    TORCH_CHECK(X.size(1) == W.size(1), "K dimension mismatch");

    int M = X.size(0);
    int K = X.size(1);
    int N = W.size(0);

    TORCH_CHECK(K % WMMA_K == 0, "K must be divisible by 16 for wmma tensor-core tiles");
    TORCH_CHECK(N % WMMA_N == 0, "N must be divisible by 16 for wmma tensor-core tiles");

    int M_padded = ((M + WMMA_M - 1) / WMMA_M) * WMMA_M;

    torch::Tensor X_padded;
    if (M_padded != M) {
        X_padded = torch::zeros({M_padded, K}, X.options());
        X_padded.slice(0, 0, M).copy_(X);
    } else {
        X_padded = X;
    }

    auto Y_padded = torch::empty({M_padded, N}, X.options().dtype(torch::kFloat32));

    dim3 block(32, NUM_WARPS, 1);   // NUM_WARPS warps cooperate on one 16x16 output tile
    dim3 grid(N / WMMA_N, M_padded / WMMA_M);

    int8_gemm_v4_kernel<<<grid, block>>>(
        X_padded.data_ptr<int8_t>(),
        W.data_ptr<int8_t>(),
        w_scale.data_ptr<float>(),
        x_scale.item<float>(),
        Y_padded.data_ptr<float>(),
        M_padded, N, K);

    return (M_padded == M) ? Y_padded : Y_padded.slice(0, 0, M).contiguous();
}

PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
    m.def("int8_gemm_v4", &int8_gemm_v4, "Multi-warp K-split tensor-core INT8 GEMM (v4)");
}

Writing kernels/int8_gemm_v4.cu


In [21]:
%%writefile benchmark.py

import time
import torch
from torch.utils.cpp_extension import load

import data
import quantize
import metrics

DEVICE = "cuda"
BATCH_SIZES = [1, 8, 32]
N_ITERS = 100
N_WARMUP = 10

print("Loading weight...")
W_fp16_cpu, source = data.load_llama_down_proj()
print(f"  source: {source}")
print(f"  shape: {tuple(W_fp16_cpu.shape)}  (Llama-3-8B mlp.down_proj: [hidden=4096, intermediate=14336])")

print("\nCompiling v1 (naive scalar) kernel...")
v1_ext = load(name="int8_gemm_ext_v1", sources=["kernels/int8_gemm_v1.cu"],
              extra_cuda_cflags=["-O3"], verbose=False)

cc_major, cc_minor = torch.cuda.get_device_capability()
print(f"GPU: {torch.cuda.get_device_name()} (compute capability {cc_major}.{cc_minor})")
if (cc_major, cc_minor) < (6, 1):
    raise RuntimeError(
        f"__dp4a requires compute capability >= 6.1; this GPU is {cc_major}.{cc_minor}. "
        f"Kaggle's P100 is CC 6.0 and does NOT support dp4a -- pick the T4 x2 accelerator "
        f"instead (Settings -> Accelerator -> GPU T4 x2). v1 (scalar, no dp4a) still runs "
        f"fine on P100 if you only want that comparison."
    )

print("Compiling v2 (dp4a vectorized) kernel...")

v2_ext = load(name="int8_gemm_ext_v2", sources=["kernels/int8_gemm_v2.cu"],
              extra_cuda_cflags=["-O3"], verbose=False)

if (cc_major, cc_minor) < (7, 5):
    raise RuntimeError(
        f"int8 wmma tensor-core tiles require compute capability >= 7.5 (Turing); "
        f"this GPU is {cc_major}.{cc_minor}. v3 needs Kaggle's T4 accelerator."
    )

print("Compiling v3 (wmma tensor-core) kernel...")
v3_ext = load(name="int8_gemm_ext_v3", sources=["kernels/int8_gemm_v3.cu"],
              extra_cuda_cflags=["-O3"], verbose=False)

print("Compiling v4 (multi-warp K-split tensor-core) kernel...")
v4_ext = load(name="int8_gemm_ext_v4", sources=["kernels/int8_gemm_v4.cu"],
              extra_cuda_cflags=["-O3"], verbose=False)

HAS_CUBLASLT_INT8 = False
try:
    _probe_a = torch.randint(-127, 127, (32, 32), dtype=torch.int8, device=DEVICE)
    _probe_b = torch.randint(-127, 127, (32, 32), dtype=torch.int8, device=DEVICE)
    torch._int_mm(_probe_a, _probe_b)
    HAS_CUBLASLT_INT8 = True
    print(f"cuBLASLt INT8 (torch._int_mm) available -- torch {torch.__version__} "
          f"(note: requires M > 16, so it will be skipped per-batch below for M<=16)")
except Exception as e:
    print(f"cuBLASLt INT8 (torch._int_mm) NOT available at all ({type(e).__name__}: {e}). "
          f"Skipping that comparison entirely -- likely needs a newer torch build. "
          f"v1-v4 vs FP16 still run below.")

def timeit(fn, n_iters=N_ITERS, n_warmup=N_WARMUP):
    for _ in range(n_warmup):
        out = fn()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n_iters):
        out = fn()
    torch.cuda.synchronize()
    return (time.perf_counter() - t0) / n_iters * 1000, out
W_fp16 = W_fp16_cpu.to(DEVICE)
t0 = time.perf_counter()
W_int8, w_scale = quantize.quantize_weight_per_channel(W_fp16)
torch.cuda.synchronize()
weight_quant_ms = (time.perf_counter() - t0) * 1000
W_int8 = W_int8.to(DEVICE).contiguous()
w_scale = w_scale.to(DEVICE).float()
print(f"\nOne-time weight quantization: {weight_quant_ms:.2f} ms (paid once at load, not per-call)")
if HAS_CUBLASLT_INT8:
    W_int8_T = W_int8.t().contiguous()   # [K, N]

IN_FEATURES = W_fp16.shape[1]
OUT_FEATURES = W_fp16.shape[0]

torch.manual_seed(0)

print(f"\nShape: X[M, {IN_FEATURES}] @ W[{OUT_FEATURES}, {IN_FEATURES}]^T -> Y[M, {OUT_FEATURES}]")
print("=" * 100)

for M in BATCH_SIZES:
    X_fp16 = (torch.randn(M, IN_FEATURES, device=DEVICE) * 0.5).half()

    fp16_ms, Y_fp16 = timeit(lambda: torch.matmul(X_fp16, W_fp16.t()))
    quant_ms, (X_int8, x_scale) = timeit(lambda: quantize.quantize_activation_dynamic(X_fp16))
    X_int8 = X_int8.to(DEVICE).contiguous()
    x_scale = x_scale.to(DEVICE).float()
    v1_gemm_ms, Y_v1 = timeit(lambda: v1_ext.int8_gemm_v1(X_int8, W_int8, w_scale, x_scale))
    v2_gemm_ms, Y_v2 = timeit(lambda: v2_ext.int8_gemm_v2(X_int8, W_int8, w_scale, x_scale))
    v3_gemm_ms, Y_v3 = timeit(lambda: v3_ext.int8_gemm_v3(X_int8, W_int8, w_scale, x_scale))
    v4_gemm_ms, Y_v4 = timeit(lambda: v4_ext.int8_gemm_v4(X_int8, W_int8, w_scale, x_scale))

    cublaslt_runs_here = HAS_CUBLASLT_INT8 and M > 16
    if cublaslt_runs_here:
   
        cublaslt_gemm_ms, cublaslt_acc = timeit(lambda: torch._int_mm(X_int8, W_int8_T))
        Y_cublaslt = quantize.dequantize(cublaslt_acc, x_scale, w_scale)   # untimed, for fidelity only
    def v1_end_to_end():
        xq, xs = quantize.quantize_activation_dynamic(X_fp16)
        return v1_ext.int8_gemm_v1(xq.to(DEVICE), W_int8, w_scale, xs.to(DEVICE).float())

    def v2_end_to_end():
        xq, xs = quantize.quantize_activation_dynamic(X_fp16)
        return v2_ext.int8_gemm_v2(xq.to(DEVICE), W_int8, w_scale, xs.to(DEVICE).float())

    def v3_end_to_end():
        xq, xs = quantize.quantize_activation_dynamic(X_fp16)
        return v3_ext.int8_gemm_v3(xq.to(DEVICE), W_int8, w_scale, xs.to(DEVICE).float())

    def v4_end_to_end():
        xq, xs = quantize.quantize_activation_dynamic(X_fp16)
        return v4_ext.int8_gemm_v4(xq.to(DEVICE), W_int8, w_scale, xs.to(DEVICE).float())

    v1_e2e_ms, _ = timeit(v1_end_to_end)
    v2_e2e_ms, _ = timeit(v2_end_to_end)
    v3_e2e_ms, _ = timeit(v3_end_to_end)
    v4_e2e_ms, _ = timeit(v4_end_to_end)

    if cublaslt_runs_here:
        def cublaslt_end_to_end():
            xq, xs = quantize.quantize_activation_dynamic(X_fp16)
            xq, xs = xq.to(DEVICE), xs.to(DEVICE).float()
            acc = torch._int_mm(xq, W_int8_T)
            return quantize.dequantize(acc, xs, w_scale)
        cublaslt_e2e_ms, _ = timeit(cublaslt_end_to_end)

    m_padded = ((M + 15) // 16) * 16
    pad_note = f"  (tensor cores pad M={M} -> {m_padded}; {1 - M / m_padded:.0%} of that tile is wasted work)" if m_padded != M else ""

    print(f"\nBatch M={M}{pad_note}")
    print(f"  FP16 (cuBLAS) baseline            : {fp16_ms:8.4f} ms")
    print(f"  activation quantization overhead  : {quant_ms:8.4f} ms")
    print(f"  v1 GEMM-only   : {v1_gemm_ms:8.4f} ms  ({fp16_ms / v1_gemm_ms:5.2f}x vs FP16)")
    print(f"  v1 quant+GEMM  : {v1_e2e_ms:8.4f} ms  ({fp16_ms / v1_e2e_ms:5.2f}x vs FP16)")
    print(f"  v2 GEMM-only   : {v2_gemm_ms:8.4f} ms  ({fp16_ms / v2_gemm_ms:5.2f}x vs FP16)")
    print(f"  v2 quant+GEMM  : {v2_e2e_ms:8.4f} ms  ({fp16_ms / v2_e2e_ms:5.2f}x vs FP16)  ")
    print(f"  v3 GEMM-only   : {v3_gemm_ms:8.4f} ms  ({fp16_ms / v3_gemm_ms:5.2f}x vs FP16)")
    print(f"  v3 quant+GEMM  : {v3_e2e_ms:8.4f} ms  ({fp16_ms / v3_e2e_ms:5.2f}x vs FP16) ")
    print(f"  v4 GEMM-only   : {v4_gemm_ms:8.4f} ms  ({fp16_ms / v4_gemm_ms:5.2f}x vs FP16)")
    print(f"  v4 quant+GEMM  : {v4_e2e_ms:8.4f} ms  ({fp16_ms / v4_e2e_ms:5.2f}x vs FP16) ")
    if cublaslt_runs_here:
        print(f"  cuBLASLt(int8) GEMM-only  : {cublaslt_gemm_ms:8.4f} ms  ({fp16_ms / cublaslt_gemm_ms:5.2f}x vs FP16)  <- real NVIDIA int8 tensor-core reference")
        print(f"  cuBLASLt(int8) quant+GEMM : {cublaslt_e2e_ms:8.4f} ms  ({fp16_ms / cublaslt_e2e_ms:5.2f}x vs FP16)")
        print(f"  v4 vs cuBLASLt(int8) GEMM-only: {cublaslt_gemm_ms / v4_gemm_ms:5.2f}x  (>1.0 means v4 beat NVIDIA's own int8 tensor-core path, not just FP16)")
    elif HAS_CUBLASLT_INT8:
        print(f"  cuBLASLt(int8): skipped -- torch._int_mm requires M > 16, this batch is M={M}")
    print(f"  fidelity vs FP16 reference:")
    metrics.print_report("v1", Y_fp16.float(), Y_v1)
    metrics.print_report("v2", Y_fp16.float(), Y_v2)
    metrics.print_report("v3", Y_fp16.float(), Y_v3)
    metrics.print_report("v4", Y_fp16.float(), Y_v4)
    if cublaslt_runs_here:
        metrics.print_report("cuBLASLt(int8)", Y_fp16.float(), Y_cublaslt)

print("\n" + "=" * 100)

if HAS_CUBLASLT_INT8:
    print()
    print("cuBLASLt(int8) is the real comparison for the 'did you beat cuBLAS' claim --")
    print("it's NVIDIA's own int8 tensor-core path (torch._int_mm, via cuBLASLt IMMA),")
    print("not the generic FP16 GEMM the rest of this table is measured against. If v4's")
    print("'vs cuBLASLt(int8)' ratio is >1.0x, that's the number worth quoting publicly --")
    print("it's a much stronger and much more specific claim than 'beat cuBLAS'.")

Overwriting benchmark.py


In [22]:
!python benchmark.py

Loading weight...
  source: real:NousResearch/Meta-Llama-3-8B:model.layers.0.mlp.down_proj.weight
  shape: (4096, 14336)  (Llama-3-8B mlp.down_proj: [hidden=4096, intermediate=14336])

Compiling v1 (naive scalar) kernel...
GPU: Tesla T4 (compute capability 7.5)
Compiling v2 (dp4a vectorized) kernel...
Compiling v3 (wmma tensor-core) kernel...
Compiling v4 (multi-warp K-split tensor-core) kernel...
cuBLASLt INT8 (torch._int_mm) available -- torch 2.10.0+cu128 (note: requires M > 16, so it will be skipped per-batch below for M<=16)

One-time weight quantization: 80.07 ms (paid once at load, not per-call)

Shape: X[M, 14336] @ W[4096, 14336]^T -> Y[M, 4096]

Batch M=1  (tensor cores pad M=1 -> 16; 94% of that tile is wasted work)
  FP16 (cuBLAS) baseline            :   0.4665 ms
  activation quantization overhead  :   0.1360 ms
  v1 GEMM-only   :   3.8535 ms  ( 0.12x vs FP16)
  v1 quant+GEMM  :   3.5129 ms  ( 0.13x vs FP16)
  v2 GEMM-only   :   0.8353 ms  ( 0.56x vs FP16)
  v2 quant+GEMM 